# Revisi ronde 3b — Kalibrasi gabungan metode skor silver + θ (validasi saja, teks identik dengan loop pelabelan)

* Hanya 76 pasangan gold **validasi** bersentimen yang dipakai untuk memilih. `test_pairs.csv` tidak dimuat. Baris gold uji ikut
  terbaca dari `gold_labeled.csv` (satu file untuk 800 baris) tetapi langsung disaring ke 60 tweet validasi; tidak ada perhitungan pada
  baris uji. Di luar validasi, yang dipakai hanya daftar `tweet_id` gold (`gold_tweet_ids.csv`) untuk mengecualikan tweet gold dari silver.
* Kalibrasi menjalankan **sel loop pelabelan silver asli** (`notebooks/02_Silverrr_Labeling_dan_SentencePair.ipynb`) tanpa perubahan:
  deteksi aspek pada `text` (teks asli), skor pada `text_v3`, label `tidak_disebutkan` bila aspek tidak terdeteksi.
  Varian `score_sentence` dibuat dengan mengganti fungsi skor yang dipanggil loop (`score_windowed` → pembungkus `score_sentence`),
  tanpa mengubah satu baris pun di loop.
* Tidak ada file lama yang diubah, tidak ada pelatihan, tidak ada panggilan API.

In [ ]:
# ── Sel 1: konfigurasi ──
import os, re, json, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
OUT_DIR = Path.cwd() if Path.cwd().name == 'revisi_round3' else Path.cwd() / 'revisi_round3'
BASE_DIR = OUT_DIR.parent
SILVER_NB = BASE_DIR / 'notebooks' / '02_Silverrr_Labeling_dan_SentencePair.ipynb'
THETAS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.0, 10.0, 15.0]
METHODS = ['score_windowed', 'score_sentence']          # urutan = prioritas bila seri
CLASSES = ['positif', 'negatif', 'netral']
ASPECTS = ['pola_konsumsi', 'aktivitas_fisik', 'akses_layanan_kesehatan', 'pengetahuan_pengobatan']
CURRENT = ('score_windowed', 3.0)                       # konfigurasi label latih saat ini
print('OUT_DIR:', OUT_DIR)

In [ ]:
# ── Sel 2: muat data (TANPA data uji) + assertion ──
def code_cells(p):
    nb = json.loads(Path(p).read_text(encoding='utf-8'))
    return [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']

def find_cell(p, *needles):
    hits = [s for s in code_cells(p) if all(n in s for n in needles)]
    assert len(hits) == 1, (needles, len(hits))
    return hits[0]

df = pd.read_csv(BASE_DIR / 'data/processed/tweets_preprocessed.csv'); df['tweet_id'] = df['tweet_id'].astype(str)
gold = pd.read_csv(BASE_DIR / 'data/gold/gold_labeled.csv'); gold['tweet_id'] = gold['tweet_id'].astype(str)
dev = pd.read_csv(BASE_DIR / 'data/pairs/dev_pairs.csv', dtype={'tweet_id': str})
gold_ids_all = set(pd.read_csv(BASE_DIR / 'data/gold/gold_tweet_ids.csv', dtype=str).tweet_id)   # hanya ID, untuk eksklusi
silver_file = pd.read_csv(BASE_DIR / 'data/silver/silver_labeled.csv'); silver_file['tweet_id'] = silver_file['tweet_id'].astype(str)
dev_ids = set(dev.tweet_id)
assert len(dev_ids) == 60 and dev_ids <= gold_ids_all and len(gold_ids_all) == 200
gold_dev = gold[gold.tweet_id.isin(dev_ids)].copy()               # hanya baris gold validasi yang dipakai
gold_dev_sent = gold_dev[gold_dev.label_gold != 'tidak_disebutkan'].copy()
assert len(gold_dev) == 240 and len(gold_dev_sent) == 76
assert not (set(silver_file.tweet_id) & gold_ids_all) and len(silver_file) == 150_988
print(f'Validasi: {len(dev_ids)} tweet, {len(gold_dev)} pasangan, {len(gold_dev_sent)} pasangan bersentimen. Baris gold uji disaring; test_pairs.csv tidak dimuat.')

## 1. Eksekusi sel asli dan pembuktian bahwa kalibrasi = loop pelabelan

In [ ]:
CELLS = {
    'lexicon':  find_cell(SILVER_NB, 'def load_lexicon'),
    'scorer':   find_cell(SILVER_NB, 'def score_sentence', 'def score_to_label', 'NEGATION_WINDOW = 2'),
    'windowed': find_cell(SILVER_NB, 'def score_windowed'),
    'keywords': find_cell(SILVER_NB, 'def detect_aspect', 'ASPECT_KEYWORDS = json.load'),
    'labeling': find_cell(SILVER_NB, 'silver_rows = []', 'THRESHOLD = BEST_THETA'),
}
LABELING_SHA = hashlib.sha256(CELLS['labeling'].encode('utf-8')).hexdigest()
base = {'os': os, 're': re, 'json': json, 'pd': pd, 'np': np, 'tqdm': tqdm, 'ASPECTS': ASPECTS,
        'LEXICON_DIR': os.path.join(BASE_DIR, 'resources', 'inset_lexicon'),
        'KEYWORDS_PATH': os.path.join(BASE_DIR, 'resources', 'aspect_keywords.json')}
for k in ['lexicon', 'scorer', 'windowed', 'keywords']:
    exec(CELLS[k], base)
print(f'Sel loop pelabelan asli (sha256 {LABELING_SHA[:16]}…):\n' + '-' * 70 + '\n' + CELLS['labeling'])

def run_labeling_loop(tweets: pd.DataFrame, method: str, theta: float) -> pd.DataFrame:
    """Eksekusi sel loop pelabelan ASLI apa adanya. Untuk method='score_sentence', fungsi skor yang dipanggil loop
    diganti menjadi score_sentence(text) (aspek diabaikan) — fungsi negasi/leksikon sama."""
    ns = dict(base)
    ns['df_for_silver'] = tweets.copy()
    ns['BEST_THETA'] = theta
    if method == 'score_sentence':
        ns['score_windowed'] = lambda text, aspect, window=5: base['score_sentence'](text)
    else:
        assert method == 'score_windowed'
    exec(CELLS['labeling'], ns)
    return ns['df_silver'][['tweet_id', 'aspek', 'label_silver', 'lexicon_score']].copy()

# Bukti 1: loop yang sama, konfigurasi saat ini, mereproduksi silver_labeled.csv 100%
df_for_silver = df[~df.tweet_id.isin(gold_ids_all)].copy()
cur = run_labeling_loop(df_for_silver, *CURRENT)
chk = cur.merge(silver_file, on=['tweet_id', 'aspek'], suffixes=('_ulang', '_file'))
proof1 = float((chk.label_silver_ulang == chk.label_silver_file).mean())
proof1s = float(np.isclose(chk.lexicon_score_ulang, chk.lexicon_score_file).mean())
assert len(chk) == 150_988 and proof1 == 1.0 and proof1s == 1.0
print(f'Bukti 1: loop pelabelan (score_windowed, θ=3,0) = silver_labeled.csv → label {proof1:.2%}, skor {proof1s:.2%}')

## 2. Kalibrasi pada validasi (loop pelabelan asli)

In [ ]:
dev_tweets = df[df.tweet_id.isin(dev_ids)].copy()
assert len(dev_tweets) == 60

# Bukti 2: skor yang dipakai kalibrasi = skor fungsi pada text_v3 (deteksi pada text), per pasangan
probe = run_labeling_loop(dev_tweets, 'score_windowed', 3.0).merge(gold_dev_sent[['tweet_id', 'aspek', 'label_gold']], on=['tweet_id', 'aspek'])
tv = dev_tweets.set_index('tweet_id')
manual = probe.apply(lambda r: base['score_windowed'](str(tv.loc[r.tweet_id, 'text_v3']), r.aspek)
                     if base['detect_aspect'](str(tv.loc[r.tweet_id, 'text']), r.aspek) else 0.0, axis=1)
proof2 = bool(np.allclose(manual.round(4), probe.lexicon_score))
assert proof2 and len(probe) == 76
print(f'Bukti 2: skor kalibrasi = score_windowed(text_v3) bila detect_aspect(text) → {proof2} (76 pasangan)')

rows = []
for method in METHODS:
    for th in THETAS:
        lab = run_labeling_loop(dev_tweets, method, th).merge(gold_dev_sent[['tweet_id', 'aspek', 'label_gold']], on=['tweet_id', 'aspek'])
        assert len(lab) == 76
        yp, yt = lab.label_silver, lab.label_gold
        vc = yp.value_counts(normalize=True) * 100
        rows.append({'metode': method, 'theta': th, 'macro_f1_validasi': f1_score(yt, yp, labels=CLASSES, average='macro', zero_division=0),
                     'akurasi_validasi': float((yt == yp).mean()), 'n_pasangan': len(lab),
                     'pred_tidak_terdeteksi': int((yp == 'tidak_disebutkan').sum()),
                     'pct_pos': vc.get('positif', 0), 'pct_neg': vc.get('negatif', 0), 'pct_net': vc.get('netral', 0)})
grid = pd.DataFrame(rows)

# Aturan pemilihan: max macro-F1; seri → score_windowed dulu, lalu θ terkecil
grid['_m'] = grid.metode.map({m: i for i, m in enumerate(METHODS)})
best_f1 = grid.macro_f1_validasi.max()
ties = grid[np.isclose(grid.macro_f1_validasi, best_f1, rtol=0, atol=1e-12)].sort_values(['_m', 'theta'])
BEST = (ties.iloc[0].metode, float(ties.iloc[0].theta))
grid['terpilih'] = (grid.metode == BEST[0]) & (grid.theta == BEST[1])
grid = grid.drop(columns='_m')
print(f'Kombinasi terpilih: {BEST[0]}, θ = {BEST[1]} (macro-F1 validasi {best_f1:.4f}); kandidat seri: {len(ties)}')
grid.pivot(index='theta', columns='metode', values='macro_f1_validasi').round(4)

## 3. Transparansi: kalibrasi gaya lama (tanpa deteksi aspek, skor pada `text`=text_v1 dari gold_labeled.csv)
Sama dengan sel sweep asli (yang menghasilkan θ = 5,0 pada validasi di `theta_report.md`), diperluas ke dua metode dan θ = 15,0.

In [ ]:
rows = []
for method in METHODS:
    for th in THETAS:
        if method == 'score_windowed':
            yp = gold_dev_sent.apply(lambda r: base['score_to_label'](base['score_windowed'](str(r['text']), r['aspek']), threshold=th), axis=1)
        else:
            yp = gold_dev_sent['text'].apply(lambda t: base['score_to_label'](base['score_sentence'](str(t)), threshold=th))
        rows.append({'metode': method, 'theta': th,
                     'macro_f1_validasi_text_v1': f1_score(gold_dev_sent.label_gold, yp, labels=CLASSES, average='macro', zero_division=0)})
grid_v1 = pd.DataFrame(rows)
old = pd.read_csv(OUT_DIR / 'theta_validation_only.csv')       # hasil ronde sebelumnya (score_windowed, text_v1)
cmp_old = grid_v1[grid_v1.metode == 'score_windowed'].merge(old[['theta', 'macro_f1_validasi']], on='theta')
assert np.allclose(cmp_old.macro_f1_validasi_text_v1, cmp_old.macro_f1_validasi), 'tidak cocok dengan theta_validation_only.csv'
b1 = grid_v1.loc[grid_v1.macro_f1_validasi_text_v1.idxmax()]
print(f'Gaya lama (text_v1): terbaik {b1.metode}, θ = {b1.theta} ({b1.macro_f1_validasi_text_v1:.4f}); score_windowed cocok dengan ronde sebelumnya.')

full = grid.merge(grid_v1, on=['metode', 'theta'])
full.to_csv(OUT_DIR / 'method_theta_validation_grid.csv', index=False, encoding='utf-8-sig')
full.round(4)

## 3b. Stabilitas pilihan (validasi saja; TIDAK dipakai untuk memilih)
* Cluster bootstrap per tweet validasi (B = 10.000): seberapa sering tiap kombinasi terpilih dengan aturan yang sama, dan CI selisih
  macro-F1 kombinasi terpilih terhadap pesaing terdekat serta terhadap konfigurasi saat ini.
* Diagnostik tepi grid: macro-F1 validasi untuk θ di luar daftar kandidat (20, 25, 30, 50). Hanya untuk menilai apakah θ terpilih berada di
  tepi grid; aturan pemilihan tidak diubah.

In [ ]:
SEED, B_BOOT = 42, 10_000
ci_ = {c: i for i, c in enumerate(CLASSES)}
def labels_for(method, th, tweets=dev_tweets):
    return run_labeling_loop(tweets, method, th).merge(gold_dev_sent[['tweet_id', 'aspek', 'label_gold']], on=['tweet_id', 'aspek'])
combos = [(m, t) for m in METHODS for t in THETAS]
ref = labels_for(*combos[0])
ids, codes = np.unique(ref.tweet_id, return_inverse=True)
yt = ref.label_gold.map(ci_).values
C = np.zeros((len(combos), len(ids), 3, 4))                   # kolom ke-4 = tidak_disebutkan (prediksi, selalu salah)
for j, (m, t) in enumerate(combos):
    lab = labels_for(m, t); assert (lab.tweet_id.values == ref.tweet_id.values).all() and (lab.aspek.values == ref.aspek.values).all()
    yp = lab.label_silver.map({**ci_, 'tidak_disebutkan': 3}).values
    np.add.at(C, (j, codes, yt, yp), 1)

def macro_f1(cm):                                            # labels = 3 kelas; prediksi tidak_disebutkan = salah (fn)
    tp = np.stack([cm[..., k, k] for k in range(3)], -1)
    fp = cm[..., :, :3].sum(-2) - tp
    fn = cm.sum(-1) - tp
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0).mean(-1)

pt = macro_f1(C.sum(1))
assert np.allclose(pt, grid.macro_f1_validasi.values), 'versi vektor ≠ f1_score'
rng = np.random.default_rng(SEED)
Wb = rng.multinomial(len(ids), np.full(len(ids), 1 / len(ids)), size=B_BOOT)
BF = macro_f1(np.einsum('bc,jcik->bjik', Wb, C))                # (B, combos)
# aturan pemilihan pada tiap resample: max; seri → urutan combos (windowed dulu, θ kecil dulu) = argmax indeks pertama
sel = pd.Series([combos[i] for i in BF.argmax(1)]).value_counts(normalize=True)
stab = pd.DataFrame({'metode': [m for m, _ in combos], 'theta': [t for _, t in combos], 'macro_f1_validasi': pt,
                     'frekuensi_terpilih_bootstrap': [sel.get(c, 0.0) for c in combos]})
ib = combos.index(BEST)
order_pt = np.argsort(-pt)
runner = combos[[i for i in order_pt if i != ib][0]]
def diff_ci(a, b):
    d = BF[:, combos.index(a)] - BF[:, combos.index(b)]
    lo, hi = np.percentile(d, [2.5, 97.5]); return float(pt[combos.index(a)] - pt[combos.index(b)]), lo, hi
D_RUNNER = diff_ci(BEST, runner)
D_CURRENT = diff_ci(BEST, CURRENT)
stab.to_csv(OUT_DIR / 'method_theta_bootstrap_selection.csv', index=False, encoding='utf-8-sig')
print(f'Terpilih {BEST}: dipilih ulang pada {sel.get(BEST, 0):.1%} resample. Pesaing terdekat {runner}: '
      f'selisih {D_RUNNER[0]:+.4f} [{D_RUNNER[1]:+.4f}; {D_RUNNER[2]:+.4f}]. vs saat ini {CURRENT}: {D_CURRENT[0]:+.4f} [{D_CURRENT[1]:+.4f}; {D_CURRENT[2]:+.4f}]')

diag = []
for th in [15.0, 20.0, 25.0, 30.0, 50.0]:
    lab = labels_for(BEST[0], th)
    diag.append({'metode': BEST[0], 'theta': th, 'macro_f1_validasi': f1_score(lab.label_gold, lab.label_silver, labels=CLASSES, average='macro', zero_division=0),
                 'pct_netral_prediksi': float((lab.label_silver == 'netral').mean() * 100), 'dalam_grid': th in THETAS})
diag = pd.DataFrame(diag)
diag.to_csv(OUT_DIR / 'method_theta_boundary_diagnostic.csv', index=False, encoding='utf-8-sig')
display(stab.sort_values('frekuensi_terpilih_bootstrap', ascending=False).head(8).round(4)); display(diag.round(4))

## 4. Dampak kombinasi terpilih pada 150.988 label latih (dibanding label saat ini)

In [ ]:
new = run_labeling_loop(df_for_silver, *BEST)
ch = cur.merge(new, on=['tweet_id', 'aspek'], suffixes=('_saat_ini', '_baru'))
assert len(ch) == 150_988
changed = ch.label_silver_saat_ini != ch.label_silver_baru
order = ['tidak_disebutkan', 'positif', 'negatif', 'netral']
trans = pd.crosstab(ch.label_silver_saat_ini, ch.label_silver_baru).reindex(index=order, columns=order, fill_value=0)
dist = pd.DataFrame({'saat_ini': ch.label_silver_saat_ini.value_counts(), 'baru': ch.label_silver_baru.value_counts()}).reindex(order).fillna(0).astype(int)
dist['selisih'] = dist.baru - dist.saat_ini
per_aspect = ch.assign(berubah=changed).groupby('aspek').berubah.agg(n_berubah='sum', proporsi='mean')
n_sent_cur = int((ch.label_silver_saat_ini != 'tidak_disebutkan').sum())
CHG = {'n': int(changed.sum()), 'p': float(changed.mean()), 'n_sent': n_sent_cur}
trans.to_csv(OUT_DIR / 'final_silver_transition.csv', encoding='utf-8-sig')
dist.to_csv(OUT_DIR / 'final_silver_distribution.csv', encoding='utf-8-sig')
per_aspect.to_csv(OUT_DIR / 'final_silver_changes_per_aspect.csv', encoding='utf-8-sig')
print(f"Label berubah: {CHG['n']:,} dari 150.988 ({CHG['p']:.2%})")
display(trans); display(dist); display(per_aspect)

## 5. Laporan `theta_final_report.md`

In [ ]:
def f(x, n=4): return f'{x:.{n}f}'.replace('.', ',')
def g(n): return f'{int(n):,}'.replace(',', '.')
pv = full.pivot(index='theta', columns='metode', values='macro_f1_validasi')
pv1 = full.pivot(index='theta', columns='metode', values='macro_f1_validasi_text_v1')
tbl = '\n'.join(
    f"| {f(t, 1)} | {f(pv.loc[t, 'score_windowed'])}{' **←**' if BEST == ('score_windowed', t) else ''} | "
    f"{f(pv.loc[t, 'score_sentence'])}{' **←**' if BEST == ('score_sentence', t) else ''} | "
    f"{f(pv1.loc[t, 'score_windowed'])} | {f(pv1.loc[t, 'score_sentence'])} |" for t in THETAS)
undet = int(full.loc[full.terpilih, 'pred_tidak_terdeteksi'].iloc[0])
tr_md = '| saat ini \\\\ baru | ' + ' | '.join(order) + ' |\n|' + '---|' * 5 + '\n' + \
    '\n'.join(f'| {i} | ' + ' | '.join(g(v) for v in r.values) + ' |' for i, r in trans.iterrows())
di_md = '| label | saat ini (windowed, θ=3,0) | baru | selisih |\n|---|---|---|---|\n' + \
    '\n'.join(f"| {i} | {g(r.saat_ini)} | {g(r.baru)} | {'+' if r.selisih >= 0 else '−'}{g(abs(r.selisih))} |" for i, r in dist.iterrows())
same_as_current = BEST == CURRENT
md = f"""# Laporan akhir kalibrasi metode skor silver dan θ (validasi saja)

Dibuat oleh `revisi_round3/calibrate_method_theta_validation.ipynb`. Hanya 76 pasangan gold validasi bersentimen (60 tweet) yang dipakai
untuk memilih. `test_pairs.csv` tidak dimuat; baris gold uji di `gold_labeled.csv` ikut terbaca lalu langsung disaring, tanpa perhitungan
apa pun pada data uji. File lama tidak diubah; tidak ada pelatihan; tidak ada API.

## 1. Kesamaan dengan loop pelabelan silver
Kalibrasi menjalankan **sel loop pelabelan asli** dari `notebooks/02_Silverrr_Labeling_dan_SentencePair.ipynb` tanpa perubahan
(sha256 sel `{LABELING_SHA[:16]}…`): deteksi aspek `detect_aspect` pada kolom `text` (teks asli); skor pada `text_v3`; `tidak_disebutkan`
bila aspek tidak terdeteksi; label `score_to_label(skor, θ)`. Varian `score_sentence` dibuat dengan mengganti fungsi skor yang dipanggil loop,
dengan leksikon dan aturan negasi yang sama.
* **Bukti 1:** loop yang sama dengan `score_windowed`, θ = 3,0 pada {g(len(df_for_silver))} tweet non-gold mereproduksi
  `data/silver/silver_labeled.csv` **{f(proof1 * 100, 2)}%** (label) dan {f(proof1s * 100, 2)}% (skor) dari 150.988 pasangan.
* **Bukti 2:** pada 76 pasangan validasi, skor yang dipakai kalibrasi sama dengan `score_windowed(text_v3)` bila `detect_aspect(text)` benar
  (dan 0 bila tidak) — {'cocok' if proof2 else 'TIDAK cocok'}.
* Perbedaan dengan sweep lama: sweep lama menilai `text_v1` dan **tanpa** deteksi aspek. Pada kalibrasi ini {undet} dari 76 pasangan
  bersentimen tidak terdeteksi aspeknya oleh kata kunci, sehingga diberi `tidak_disebutkan` dan selalu dihitung salah, persis seperti
  yang akan terjadi pada data latih.

## 2. Hasil kalibrasi (macro-F1 3 kelas, 76 pasangan validasi)
Aturan: macro-F1 tertinggi; bila seri, `score_windowed` lebih dulu, lalu θ yang lebih kecil.

| θ | score_windowed (loop, text_v3) | score_sentence (loop, text_v3) | score_windowed (gaya lama, text_v1) | score_sentence (gaya lama, text_v1) |
|---|---|---|---|---|
{tbl}

**Kombinasi terpilih: `{BEST[0]}`, θ = {f(BEST[1], 1)}** (macro-F1 validasi {f(best_f1)}; {len(ties)} kandidat dengan skor tertinggi).
{'Sama dengan konfigurasi label latih saat ini.' if same_as_current else f'Berbeda dari konfigurasi label latih saat ini (score_windowed, θ = 3,0).'}

Transparansi (gaya lama, `text_v1`, tanpa deteksi): terbaik `{b1.metode}`, θ = {f(b1.theta, 1)} (macro-F1 {f(b1.macro_f1_validasi_text_v1)}).
Kolom `score_windowed` gaya lama identik dengan `theta_validation_only.csv` ronde sebelumnya (yang menghasilkan θ = 5,0).
Tabel lengkap (termasuk akurasi dan distribusi prediksi): `method_theta_validation_grid.csv`.

### Stabilitas pilihan (validasi saja; tidak dipakai untuk memilih)
* Cluster bootstrap per tweet validasi ({len(ids)} tweet, B = 10.000), aturan pemilihan yang sama: kombinasi terpilih muncul lagi pada
  **{f(sel.get(BEST, 0) * 100, 1)}%** resample. Tiga kombinasi paling sering terpilih: """ + '; '.join(
    f"`{r.metode}` θ = {f(r.theta, 1)} ({f(r.frekuensi_terpilih_bootstrap * 100, 1)}%)"
    for r in stab.sort_values('frekuensi_terpilih_bootstrap', ascending=False).head(3).itertuples()) + f""".
* Selisih terhadap pesaing terdekat (`{runner[0]}`, θ = {f(runner[1], 1)}): {f(D_RUNNER[0])} [CI 95% {f(D_RUNNER[1])}; {f(D_RUNNER[2])}].
  Selisih terhadap konfigurasi saat ini (`score_windowed`, θ = 3,0): {f(D_CURRENT[0])} [{f(D_CURRENT[1])}; {f(D_CURRENT[2])}].
* **Tepi grid:** θ terpilih {'adalah kandidat terbesar (15,0)' if BEST[1] == max(THETAS) else 'bukan kandidat terbesar'}. Diagnostik di luar grid
  (tidak dipakai untuk memilih): """ + '; '.join(f"θ = {f(r.theta, 0)} → {f(r.macro_f1_validasi)} (netral {f(r.pct_netral_prediksi, 0)}%)"
                                                for r in diag.itertuples()) + f""".

## 3. Dampak pada 150.988 label latih
* **{g(CHG['n'])} label berubah ({f(CHG['p'] * 100, 2)}%)** dibanding label latih saat ini.

{tr_md}

{di_md}

Per aspek: `final_silver_changes_per_aspect.csv`.

## 4. Kesimpulan
* Metode dan θ untuk latih ulang: **`{BEST[0]}`, θ = {f(BEST[1], 1)}**, dipilih hanya dengan data validasi dan dengan teks yang identik
  dengan loop pelabelan. Keputusan ini dibekukan di `rencana_latih_ulang.md` sebelum data uji disentuh.
* Pilihan ini adalah hasil aturan yang ditetapkan sebelumnya, bukan bukti bahwa kandidat lain lebih buruk: dengan 76 pasangan validasi,
  selisih terhadap pesaing terdekat ({f(D_RUNNER[0])}) jauh di dalam ketidakpastian sampling, dan kombinasi terpilih hanya muncul lagi pada
  {f(sel.get(BEST, 0) * 100, 1)}% resample.
* **Perhatian sebelum melatih ulang:** θ terpilih berada di tepi grid, dan label latih bergeser besar ke netral
  ({g(CHG['n'])} label berubah; positif {g(dist.loc['positif', 'saat_ini'])} → {g(dist.loc['positif', 'baru'])}, negatif
  {g(dist.loc['negatif', 'saat_ini'])} → {g(dist.loc['negatif', 'baru'])}). Bila aturan pemilihan ingin diubah (misalnya memperluas grid atau
  membatasi θ), keputusan itu harus dibuat **sekarang**, hanya dengan data validasi, dan dicatat sebagai perubahan rencana. Data uji masih
  belum disentuh, jadi perubahan seperti ini belum menimbulkan kebocoran, tetapi harus diungkapkan.
"""
(OUT_DIR / 'theta_final_report.md').write_text(md, encoding='utf-8')
json.dump({'metode': BEST[0], 'theta': BEST[1], 'macro_f1_validasi': float(best_f1), 'labeling_cell_sha256': LABELING_SHA,
           'n_label_berubah': CHG['n']}, open(OUT_DIR / 'final_silver_config.json', 'w'), indent=2)
print(md)